# Lab 2: Inferencia On-Device en Web y Python con LiteRT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-source-models/blob/main/session-01-hf-kerashub-litert/02-litert-web-vision/notebook.ipynb)

## Objetivo
Comprender la estructura de los modelos compactos **LiteRT** (`ai-edge-litert`), inspeccionar sus tensores de entrada y salida, y ejecutar inferencia on-device de alto rendimiento tanto en Python como en la Web ([Google AI Edge LiteRT Web](https://developers.google.com/edge/litert/web/get_started)).

### Paso 1: Instalacion de Dependencias LiteRT

In [ ]:
!pip install -q ai-edge-litert pillow numpy requests matplotlib

### Paso 2: Descarga del Modelo Oficial LiteRT de Google AI Edge e Imagenes de Muestra

In [ ]:
import os
import urllib.request
import ssl
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs("models", exist_ok=True)
os.makedirs("sample_images", exist_ok=True)
ctx = ssl._create_unverified_context()

# Descargar modelo oficial Google AI Edge LiteRT
model_path = "models/efficientnet_lite0.tflite"
if not os.path.exists(model_path) or os.path.getsize(model_path) < 1000000:
    url = "https://storage.googleapis.com/mediapipe-models/image_classifier/efficientnet_lite0/float32/1/efficientnet_lite0.tflite"
    print("Descargando modelo LiteRT de Google...")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=ctx) as resp, open(model_path, "wb") as f:
        f.write(resp.read())
    print(f"Modelo LiteRT guardado: {model_path} ({os.path.getsize(model_path)/(1024*1024):.2f} MB)")

# Descargar etiquetas ImageNet
labels_path = "models/imagenet_classes.json"
if not os.path.exists(labels_path):
    labels_url = "https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt"
    with urllib.request.urlopen(labels_url, context=ctx) as r:
        raw_labels = r.read().decode("utf-8").strip().splitlines()
    labels = [l.strip().replace("_", " ").title() for l in raw_labels if l.strip()]
    with open(labels_path, "w", encoding="utf-8") as f:
        json.dump(labels, f, indent=2)
    print(f"Etiquetas guardadas ({len(labels)} clases).")

### Paso 3: Inicializacion del Interprete LiteRT en Python

In [ ]:
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("=== Detalles del Tensor de Entrada ===")
print(f"Nombre: {input_details[0]['name']}")
print(f"Forma : {input_details[0]['shape']}")
print(f"DType : {input_details[0]['dtype']}")

print("\n=== Detalles del Tensor de Salida ===")
print(f"Nombre: {output_details[0]['name']}")
print(f"Forma : {output_details[0]['shape']}")
print(f"DType : {output_details[0]['dtype']}")

### Paso 4: Inferencia On-Device en Python sobre Fotos Reales

In [ ]:
import time

# Descargar imagen de prueba (Golden Retriever)
dog_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b3/Golden_Retriever_2019.jpg/500px-Golden_Retriever_2019.jpg"
dog_path = "sample_images/dog.jpg"
req = urllib.request.Request(dog_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, context=ctx) as resp, open(dog_path, "wb") as f:
    f.write(resp.read())

img = Image.open(dog_path).convert("RGB").resize((224, 224))
# Normalizacion [-1.0, 1.0] para EfficientNet
img_array = (np.array(img, dtype=np.float32) / 127.5) - 1.0
input_data = np.expand_dims(img_array, axis=0)

# Ejecutar inferencia en LiteRT
interpreter.set_tensor(input_details[0]['index'], input_data)
start_time = time.time()
interpreter.invoke()
latency_ms = (time.time() - start_time) * 1000

output_data = interpreter.get_tensor(output_details[0]['index'])[0]
probs = output_data.astype(np.float32)

with open(labels_path) as f:
    labels = json.load(f)

top_indices = np.argsort(probs)[::-1][:5]
print(f"Latencia de inferencia LiteRT: {latency_ms:.2f} ms\n")
print("Top 5 Predicciones:")
for rank, idx in enumerate(top_indices, start=1):
    label_name = labels[idx] if idx < len(labels) else f"Class {idx}"
    print(f"  {rank}. {label_name:<30} {probs[idx]*100:.2f}%")

### Paso 5: Despliegue en la Web con JavaScript
Para ejecutar este mismo modelo directamente en el navegador del cliente sin backend:
1. Inicie el servidor estatico local: `npx serve . -p 3000`
2. Abra `http://localhost:3000` en su navegador.
3. La inferencia se ejecutara en WebAssembly/WebGL en el hardware del cliente.

### Paso Final: Limpieza del Interprete LiteRT y Modelos Descargados

In [ ]:
import gc, shutil, os

del interpreter
gc.collect()

if os.path.exists('sample_images'):
    shutil.rmtree('sample_images')
print('Memoria y archivos temporales liberados.')